# Notebook 5: Bracket Generator
## Generate optimal brackets for multiple pool configurations
---
**Input:** Simulation results from `results/simulation_results.csv`  
**Output:** `brackets/` directory with one CSV per pool + printable summaries

Run this AFTER Notebooks 1–3. Re-run whenever you update simulations.

In [1]:
# ============================================================
# 5.0 CONFIG & IMPORTS
# ============================================================
import os, json, warnings, logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("brackets")

RESULTS_DIR = Path("./results")
BRACKETS_DIR = Path("./brackets")
BRACKETS_DIR.mkdir(exist_ok=True)

# Load simulation results
sim_results = pd.read_csv(RESULTS_DIR / "simulation_results.csv")
log.info(f"Loaded simulation results: {sim_results.shape[0]} teams")

# ==========================================================
# POOL DEFINITIONS
# ==========================================================
POOLS = {
    "Family": {
        "scoring": "espn",
        "pool_size_low": 8, "pool_size_high": 12,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "description": "ESPN default, 8-12 people",
    },
    "Fantasy_Prem": {
        "scoring": "espn",
        "pool_size_low": 3, "pool_size_high": 6,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "description": "ESPN default, 3-5 people",
    },
    "College": {
        "scoring": "yahoo",
        "pool_size_low": 7, "pool_size_high": 10,
        "yahoo_points": [1, 2, 4, 8, 16, 32],
        "description": "Yahoo default, 6-10 people",
    },
    "Dereks": {
        "scoring": "espn",
        "pool_size_low": 18, "pool_size_high": 25,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "description": "ESPN default, 12-15 people",
    },
    "DK_Company": {
        "scoring": "espn",
        "pool_size_low": 2000, "pool_size_high": 3000,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "description": "ESPN default, 2k+ people — MAX CONTRARIAN",
    },
    "DK_ATLAS_Org": {
        "scoring": "custom_inverse",
        "pool_size_low": 25, "pool_size_high": 50,
        "description": "1 point per correct pick / number of people with that pick, 20-50 people",
    },
    "DK_Analytics_Org": {
        "scoring": "espn",
        "pool_size_low": 100, "pool_size_high": 125,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "description": "ESPN default, 75-100 people",
    },
}

print(f"Configured {len(POOLS)} pools:")
for name, cfg in POOLS.items():
    print(f"  {name}: {cfg['description']}")

2026-03-19 01:50:31,375 [INFO] Loaded simulation results: 64 teams


Configured 7 pools:
  Family: ESPN default, 8-12 people
  Fantasy_Prem: ESPN default, 3-5 people
  College: Yahoo default, 6-10 people
  Dereks: ESPN default, 12-15 people
  DK_Company: ESPN default, 2k+ people — MAX CONTRARIAN
  DK_ATLAS_Org: 1 point per correct pick / number of people with that pick, 20-50 people
  DK_Analytics_Org: ESPN default, 75-100 people


In [2]:
# ============================================================
# 5.1 CONTRARIAN & OWNERSHIP MODEL
# ============================================================
def get_contrarian_factor(pool_size_low, pool_size_high):
    """Auto-set contrarian aggressiveness based on pool size midpoint.
    
    Capped at 0.40 — historical data shows even in 2k+ pools, 
    the champion is a 1 or 2 seed ~78% of the time. Going full
    contrarian (picking 5+ seeds as champ) is -EV.
    """
    mid = (pool_size_low + pool_size_high) / 2
    if mid < 10: return 0.0       # pure chalk
    if mid < 25: return 0.05      # barely contrarian
    if mid < 75: return 0.10      # mild
    if mid < 200: return 0.20     # moderate
    if mid < 1000: return 0.30    # noticeable
    return 0.40                    # max contrarian for 1k+


def estimate_public_ownership(seed, round_name):
    """Estimate what % of a typical pool picks this seed to advance."""
    base = {
        1: {"R64": .99, "R32": .93, "S16": .80, "E8": .58, "F4": .38, "Championship": .20, "Champion": .12},
        2: {"R64": .94, "R32": .82, "S16": .58, "E8": .33, "F4": .17, "Championship": .09, "Champion": .05},
        3: {"R64": .85, "R32": .62, "S16": .38, "E8": .18, "F4": .08, "Championship": .04, "Champion": .015},
        4: {"R64": .79, "R32": .52, "S16": .28, "E8": .12, "F4": .05, "Championship": .025, "Champion": .008},
        5: {"R64": .65, "R32": .38, "S16": .18, "E8": .07, "F4": .03, "Championship": .012, "Champion": .004},
        6: {"R64": .63, "R32": .35, "S16": .14, "E8": .05, "F4": .02, "Championship": .008, "Champion": .003},
        7: {"R64": .61, "R32": .30, "S16": .12, "E8": .04, "F4": .015, "Championship": .006, "Champion": .002},
        8: {"R64": .51, "R32": .22, "S16": .08, "E8": .025, "F4": .008, "Championship": .003, "Champion": .001},
    }
    if seed <= 8:
        return base.get(seed, {}).get(round_name, 0.005)
    # Seeds 9-16: decay from seed 8
    s8 = base.get(8, {}).get(round_name, 0.005)
    return max(0.001, s8 * (0.5 ** ((seed - 8) / 4)))


def optimize_bracket(sim_results, pool_config, pool_name):
    """Generate optimal bracket picks for a specific pool."""
    scoring = pool_config["scoring"]
    cf = get_contrarian_factor(pool_config["pool_size_low"], pool_config["pool_size_high"])
    pool_mid = (pool_config["pool_size_low"] + pool_config["pool_size_high"]) / 2
    
    log.info(f"  {pool_name}: scoring={scoring}, pool~{pool_mid:.0f}, contrarian={cf:.2f}")
    
    df = sim_results.copy()
    round_names = ["R64", "R32", "S16", "E8", "F4", "Championship", "Champion"]
    
    for rnd in round_names:
        prob_col = f"prob_{rnd}"
        if prob_col not in df.columns:
            continue
            
        df[f"own_{rnd}"] = df["seed"].apply(lambda s: estimate_public_ownership(s, rnd))
        our_prob = df[prob_col]
        ownership = df[f"own_{rnd}"].clip(lower=0.001)
        
        # Probability floor: don't recommend a team for a deep run
        # if their probability is negligible (avoids absurd contrarian picks)
        prob_floor = {"R64": 0.0, "R32": 0.0, "S16": 0.05, "E8": 0.03,
                      "F4": 0.02, "Championship": 0.01, "Champion": 0.005}
        floor = prob_floor.get(rnd, 0.0)
        eligible = our_prob >= floor
        
        if scoring == "custom_inverse":
            # DK ATLAS: points = 1/ownership. Value = prob * (1/ownership)
            df[f"value_{rnd}"] = np.where(eligible, our_prob / ownership, 0)
        else:
            # ESPN and Yahoo: blend raw prob with leverage
            raw_value = (1 - cf) * our_prob + cf * (our_prob / ownership)
            df[f"value_{rnd}"] = np.where(eligible, raw_value, our_prob)
    
    return df.sort_values("value_Champion", ascending=False), cf

print("Optimization functions loaded")

Optimization functions loaded


In [3]:
# ============================================================
# 5.2 GENERATE ALL BRACKETS
# ============================================================
all_brackets = {}

for pool_name, pool_config in POOLS.items():
    bracket_df, cf = optimize_bracket(sim_results, pool_config, pool_name)
    all_brackets[pool_name] = bracket_df
    
    # Save individual bracket
    bracket_df.to_csv(BRACKETS_DIR / f"bracket_{pool_name}.csv", index=False)
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"  {pool_name}: {pool_config['description']}")
    print(f"  Contrarian factor: {cf:.2f}")
    print(f"{'='*60}")
    
    # Champion pick
    # Champion must have ≥2% championship probability
    champ_contenders = bracket_df[bracket_df["prob_Champion"] >= 0.02]
    if len(champ_contenders) == 0:
        champ_contenders = bracket_df.nlargest(5, "prob_Champion")
    champ = champ_contenders.sort_values("value_Champion", ascending=False).iloc[0]

    print(f"   CHAMPION: {champ['team']} ({int(champ['seed'])}-seed)")
    print(f"     Win prob: {champ['prob_Champion']:.1%}, Value: {champ['value_Champion']:.4f}")
    
    # Final Four
    print(f"   FINAL FOUR:")
    # Pick top team per region — must have ≥10% F4 probability to be considered
    # (historically, F4 teams are almost always seeds 1-5)
    for region in ["East", "West", "South", "Midwest"]:
        region_df = bracket_df[bracket_df["region"] == region]
        # Filter to realistic F4 contenders
        contenders = region_df[region_df["prob_F4"] >= 0.10]
        if len(contenders) == 0:
            contenders = region_df.nlargest(3, "prob_F4")  # fallback to top 3
        if "value_F4" in contenders.columns:
            top = contenders.sort_values("value_F4", ascending=False).iloc[0]
        else:
            top = contenders.sort_values("prob_F4", ascending=False).iloc[0]
        print(f"     {region:8s}: {top['team']:20s} ({int(top['seed'])}-seed, "
              f"F4 prob={top['prob_F4']:.1%})")
            
    
    # Key upsets to consider (value significantly above chalk)
    if cf > 0.1:
        print(f"   UPSET PICKS (high leverage):")
        upsets = bracket_df[
            (bracket_df["seed"] >= 5) & 
            (bracket_df["prob_S16"] > 0.1)
        ].sort_values("value_S16" if "value_S16" in bracket_df.columns else "prob_S16", 
                       ascending=False).head(5)
        for _, u in upsets.iterrows():
            print(f"     {u['team']:20s} ({int(u['seed'])}-seed) → S16: {u['prob_S16']:.1%}")

log.info(f"\nAll brackets saved to {BRACKETS_DIR}/")

2026-03-19 01:50:31,383 [INFO]   Family: scoring=espn, pool~10, contrarian=0.05
2026-03-19 01:50:31,390 [INFO]   Fantasy_Prem: scoring=espn, pool~4, contrarian=0.00
2026-03-19 01:50:31,395 [INFO]   College: scoring=yahoo, pool~8, contrarian=0.00
2026-03-19 01:50:31,400 [INFO]   Dereks: scoring=espn, pool~22, contrarian=0.05
2026-03-19 01:50:31,404 [INFO]   DK_Company: scoring=espn, pool~2500, contrarian=0.40
2026-03-19 01:50:31,409 [INFO]   DK_ATLAS_Org: scoring=custom_inverse, pool~38, contrarian=0.10
2026-03-19 01:50:31,413 [INFO]   DK_Analytics_Org: scoring=espn, pool~112, contrarian=0.20
2026-03-19 01:50:31,418 [INFO] 
All brackets saved to brackets/



  Family: ESPN default, 8-12 people
  Contrarian factor: 0.05
   CHAMPION: Michigan (1-seed)
     Win prob: 25.4%, Value: 0.3474
   FINAL FOUR:
     East    : Duke                 (1-seed, F4 prob=64.7%)
     West    : Arizona              (1-seed, F4 prob=55.0%)
     South   : Florida              (1-seed, F4 prob=38.3%)
     Midwest : Michigan             (1-seed, F4 prob=59.9%)

  Fantasy_Prem: ESPN default, 3-5 people
  Contrarian factor: 0.00
   CHAMPION: Michigan (1-seed)
     Win prob: 25.4%, Value: 0.2542
   FINAL FOUR:
     East    : Duke                 (1-seed, F4 prob=64.7%)
     West    : Arizona              (1-seed, F4 prob=55.0%)
     South   : Florida              (1-seed, F4 prob=38.3%)
     Midwest : Michigan             (1-seed, F4 prob=59.9%)

  College: Yahoo default, 6-10 people
  Contrarian factor: 0.00
   CHAMPION: Michigan (1-seed)
     Win prob: 25.4%, Value: 0.2542
   FINAL FOUR:
     East    : Duke                 (1-seed, F4 prob=64.7%)
     West    : Ari

In [4]:
# ============================================================
# 5.3 COMPARISON TABLE
# ============================================================
# Side-by-side champion + Final Four picks across all pools

comparison = []
for pool_name, bracket_df in all_brackets.items():
    champ = bracket_df.iloc[0]
    row = {
        "Pool": pool_name,
        "Champion": f"{champ['team']} ({int(champ['seed'])})",
        "Champ_Prob": f"{champ['prob_Champion']:.1%}",
    }
    for region in ["East", "West", "South", "Midwest"]:
        region_df = bracket_df[bracket_df["region"] == region]
        val_col = "value_F4" if "value_F4" in region_df.columns else "prob_F4"
        top = region_df.sort_values(val_col, ascending=False).iloc[0]
        row[f"F4_{region}"] = f"{top['team']} ({int(top['seed'])})"
    comparison.append(row)

comp_df = pd.DataFrame(comparison)
print("\n  BRACKET COMPARISON ACROSS POOLS:")
print("=" * 100)
display(comp_df)
comp_df.to_csv(BRACKETS_DIR / "comparison_summary.csv", index=False)

print(f"\n All {len(POOLS)} brackets generated in {BRACKETS_DIR}/")


  BRACKET COMPARISON ACROSS POOLS:


,Pool,Champion,Champ_Prob,F4_East,F4_West,F4_South,F4_Midwest
0,Family,Michigan (1),25.4%,Duke (1),Arizona (1),Florida (1),Michigan (1)
1,Fantasy_Prem,Michigan (1),25.4%,Duke (1),Arizona (1),Florida (1),Michigan (1)
2,College,Michigan (1),25.4%,Duke (1),Arizona (1),Florida (1),Michigan (1)
3,Dereks,Michigan (1),25.4%,Duke (1),Arizona (1),Florida (1),Michigan (1)
4,DK_Company,Illinois (3),4.8%,Ohio St. (8),Arizona (1),Illinois (3),Texas Tech (5)
5,DK_ATLAS_Org,Illinois (3),4.8%,Ohio St. (8),Wisconsin (5),Vanderbilt (5),Texas Tech (5)
6,DK_Analytics_Org,Illinois (3),4.8%,Duke (1),Arizona (1),Illinois (3),Michigan (1)



 All 7 brackets generated in brackets/


In [6]:
# ============================================================
# 5.4 PRINTABLE BRACKET PICK SHEETS
# ============================================================
# Walk the actual bracket tree game-by-game using value columns.
# For each matchup, the team with the higher value_<next_round> is picked.

bracket_2026 = pd.read_csv("./data/bracket_2026.csv")

# Standard NCAA bracket: seeds paired within each region
# R64 matchups by seed: (1,16), (8,9), (5,12), (4,13), (6,11), (3,14), (7,10), (2,15)
# R32 matchups: winner of (1,16) vs winner of (8,9), etc.
BRACKET_ORDER = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
REGION_ORDER = ["East", "West", "South", "Midwest"]
ROUND_NAMES = ["R64", "R32", "S16", "E8"]
# Value column to use for picking who advances TO that round:
# Picking R64 winners = who reaches R32 → use value_R32
# Picking R32 winners = who reaches S16 → use value_S16
# etc.
ADVANCE_VALUE = {
    "R64": "value_R32",
    "R32": "value_S16",
    "S16": "value_E8",
    "E8": "value_F4",
    "F4": "value_Championship",
    "Championship": "value_Champion",
}


def walk_bracket(bracket_df, pool_name):
    """Walk the bracket tree and pick winners game by game."""
    picks = []  # list of (round, game_num, region, winner_team, winner_seed, loser_team, loser_seed)
    
    # Build initial bracket per region
    region_teams = {}
    for region in REGION_ORDER:
        reg = bracket_df[bracket_df["region"] == region]
        ordered = []
        for seed in BRACKET_ORDER:
            team_row = reg[reg["seed"] == seed]
            if len(team_row) > 0:
                ordered.append(team_row.iloc[0])
            else:
                ordered.append(None)
        region_teams[region] = ordered
    
    # --- R64 through E8 (within regions) ---
    f4_teams = []
    for region in REGION_ORDER:
        current = region_teams[region]
        
        for rnd_idx, rnd_name in enumerate(ROUND_NAMES):
            value_col = ADVANCE_VALUE[rnd_name]
            next_round = []
            game_in_round = 0
            
            for i in range(0, len(current), 2):
                t1 = current[i]
                t2 = current[i + 1]
                
                if t1 is None or t2 is None:
                    next_round.append(t1 if t2 is None else t2)
                    continue
                
                v1 = t1.get(value_col, t1.get(f"prob_{rnd_name}", 0)) if isinstance(t1, pd.Series) else 0
                v2 = t2.get(value_col, t2.get(f"prob_{rnd_name}", 0)) if isinstance(t2, pd.Series) else 0
                
                if v1 >= v2:
                    winner, loser = t1, t2
                else:
                    winner, loser = t2, t1
                
                game_in_round += 1
                picks.append({
                    "Round": rnd_name,
                    "Region": region,
                    "Game": game_in_round,
                    "Pick": f"({int(winner['seed'])}) {winner['team']}",
                    "Over": f"({int(loser['seed'])}) {loser['team']}",
                    "Pick_Prob": f"{winner.get(f'prob_{rnd_name}', 0):.0%}" if rnd_name == "R64" else f"{winner.get('prob_' + rnd_name.replace('R64','R32'), 0):.0%}",
                })
                next_round.append(winner)
            
            current = next_round
        
        # After E8, the survivor is the F4 rep for this region
        if current and len(current) == 1:
            f4_teams.append(current[0])
    
    # --- Final Four (cross-region) ---
    if len(f4_teams) == 4:
        # F4: East vs South, West vs Midwest (standard NCAA bracket)
        f4_matchups = [(0, 2), (1, 3)]  # East vs South, West vs Midwest
        champ_game = []
        
        for t1_idx, t2_idx in f4_matchups:
            t1, t2 = f4_teams[t1_idx], f4_teams[t2_idx]
            value_col = ADVANCE_VALUE["F4"]
            v1 = t1.get(value_col, 0) if isinstance(t1, pd.Series) else 0
            v2 = t2.get(value_col, 0) if isinstance(t2, pd.Series) else 0
            
            if v1 >= v2:
                winner, loser = t1, t2
            else:
                winner, loser = t2, t1
            
            picks.append({
                "Round": "F4",
                "Region": f"{REGION_ORDER[t1_idx]} vs {REGION_ORDER[t2_idx]}",
                "Game": len([p for p in picks if p['Round'] == 'F4']) + 1,
                "Pick": f"({int(winner['seed'])}) {winner['team']}",
                "Over": f"({int(loser['seed'])}) {loser['team']}",
                "Pick_Prob": f"{winner.get('prob_Championship', 0):.0%}",
            })
            champ_game.append(winner)
        
        # --- Championship ---
        if len(champ_game) == 2:
            t1, t2 = champ_game[0], champ_game[1]
            value_col = ADVANCE_VALUE["Championship"]
            v1 = t1.get(value_col, 0) if isinstance(t1, pd.Series) else 0
            v2 = t2.get(value_col, 0) if isinstance(t2, pd.Series) else 0
            
            if v1 >= v2:
                winner, loser = t1, t2
            else:
                winner, loser = t2, t1
            
            picks.append({
                "Round": "Championship",
                "Region": "Final",
                "Game": 1,
                "Pick": f"({int(winner['seed'])}) {winner['team']}",
                "Over": f"({int(loser['seed'])}) {loser['team']}",
                "Pick_Prob": f"{winner.get('prob_Champion', 0):.0%}",
            })
    
    return pd.DataFrame(picks)


# Generate and display pick sheets for each pool
for pool_name in POOLS:
    bracket_df = all_brackets[pool_name]
    picks_df = walk_bracket(bracket_df, pool_name)
    
    # Save to file
    picks_df.to_csv(BRACKETS_DIR / f"picks_{pool_name}.csv", index=False)
    
    # Print formatted pick sheet
    print(f"\n{'#'*70}")
    print(f"  {pool_name}: {POOLS[pool_name]['description']}")
    print(f"{'#'*70}")
    
    for rnd in ["R64", "R32", "S16", "E8", "F4", "Championship"]:
        rnd_picks = picks_df[picks_df["Round"] == rnd]
        if len(rnd_picks) == 0:
            continue
        
        label = {"R64": "ROUND OF 64", "R32": "ROUND OF 32", "S16": "SWEET 16",
                 "E8": "ELITE 8", "F4": "FINAL FOUR", "Championship": "CHAMPIONSHIP"}
        print(f"\n  --- {label.get(rnd, rnd)} ---")
        
        current_region = ""
        for _, pick in rnd_picks.iterrows():
            if pick["Region"] != current_region and rnd in ["R64", "R32", "S16", "E8"]:
                current_region = pick["Region"]
                print(f"    [{current_region}]")
            print(f"      {pick['Pick']:30s} over {pick['Over']}")
    
    # Champion summary
    champ_row = picks_df[picks_df["Round"] == "Championship"]
    if len(champ_row) > 0:
        print(f"\n  >>> CHAMPION: {champ_row.iloc[0]['Pick']} <<<")
    print()

log.info(f"Pick sheets saved to {BRACKETS_DIR}/picks_*.csv")
print(f"\nTo fill out your brackets: open picks_<PoolName>.csv and follow the picks round by round.")

2026-03-19 02:02:49,046 [INFO] Pick sheets saved to brackets/picks_*.csv



######################################################################
  Family: ESPN default, 8-12 people
######################################################################

  --- ROUND OF 64 ---
    [East]
      (1) Duke                       over (16) Siena
      (8) Ohio St.                   over (9) TCU
      (5) St. John's                 over (12) Northern Iowa
      (4) Kansas                     over (13) Cal Baptist
      (6) Louisville                 over (11) South Florida
      (3) Michigan St.               over (14) North Dakota St.
      (7) UCLA                       over (10) UCF
      (2) UConn                      over (15) Furman
    [West]
      (1) Arizona                    over (16) Long Island
      (8) Villanova                  over (9) Utah St.
      (5) Wisconsin                  over (12) High Point
      (4) Arkansas                   over (13) Hawaii
      (6) BYU                        over (11) Texas
      (3) Gonzaga                    over (1

## Bracket Generator Summary

Each pool gets a tailored bracket based on:
- **Scoring system** (ESPN, Yahoo, custom inverse ownership)
- **Pool size** → contrarian factor (small pools play chalk, large pools differentiate)
- **Ownership estimates** → leverage (pick teams the public undervalues)

### Pool Strategy Guide:
| Pool | Strategy | Why |
|------|----------|-----|
| Family (8-12) | Pure chalk | Small pool, just need to be right |
| Fantasy Prem (3-5) | Pure chalk | Tiny pool, pick the best team |
| College (6-10) | Chalk | Small pool, just need to be right |
| Derek's (12-15) | Chalk | Still small enough for chalk |
| DK Company (2k+) | Max contrarian | Need differentiation to win |
| DK ATLAS (20-50) | Built-in contrarian | Scoring = 1/ownership, naturally rewards uniqueness |
| DK Analytics (75-100) | Moderate contrarian | Large enough to need some differentiation |